# Session 2. LangGraph: state, nodes, edges

**The loop becomes a shape you declare. Something else runs it.**

- same behaviour; the new owner buys pause, checkpoints, tracing, subgraphs
- `init_chat_model` in one place: no model id or key in notebooks
- no temperature anywhere: current Gemini models loop off the default


In [ ]:
import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])


## What the hand-written loop cannot do

**One tool is not a coincidence. It is the ceiling.**

- a second tool: parse a name, branch, grow a dispatch table
- no pause, no resume: the run's state is local variables in a `for`, gone when it returns
- no instrument but `print`: not which node ran, not what the model got on turn three, not how long the tool took
- `careNotes` comes back `None`: same as "the model did not call"


In [ ]:
QUOTES = "'" + '"'  # the model quotes its argument on some turns only


# one tool: the name is implied, not returned
def parse_action_one(text: str) -> str | None:
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("Action:") and "(" in line:
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip(QUOTES)
    return None  # no Action line: the model answered instead


# two tools: the return type changes, so every caller does
def parse_action_two(text: str) -> tuple[str, str] | None:
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("Action:") or "(" not in line:
            continue
        body = line[len("Action:") :].strip()
        name = body[: body.index("(")].strip()
        argument = body[body.index("(") + 1 : body.rindex(")")].strip().strip(QUOTES)
        if name not in ("find_species", "care_notes"):
            return None  # the model invented a tool. Now what?
        return name, argument
    return None


print(parse_action_two('Action: care_notes("mon-01")'))
print(parse_action_two('Action: careNotes("mon-01")'))


## Building the graph: state

**One shared dict. Every node returns a partial update.**

- the update goes to the field's reducer, not to the state
- the schema goes to the graph, not to the nodes
- every `invoke` today passes a `recursion_limit`; why, later


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph


class Broken(TypedDict):
    messages: list  # no reducer, so the default applies: replacement


def add_reply(state) -> dict:
    return {"messages": [AIMessage("the reply")]}  # never looks at the history


builder = StateGraph(Broken)
builder.add_node("add_reply", add_reply)
builder.add_edge(START, "add_reply")  # START and END are the two built-in nodes
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])


**One message. The question is gone, nothing warned.**

- you will say "the model keeps forgetting" and look at the model
- history vanishes? look here first. Most common mistake of the session
- `add_messages` matches on id, so a replayed history does not double


In [ ]:
from langgraph.graph.message import add_messages


class Fixed(TypedDict):
    messages: Annotated[list, add_messages]  # reducer(old, update) -> new; appends


builder = StateGraph(Fixed)
builder.add_node("add_reply", add_reply)  # the same node function, untouched
builder.add_edge(START, "add_reply")
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])


In [ ]:
from langgraph.graph import MessagesState

print(MessagesState.__annotations__["messages"])  # that one field, never retyped


### Nodes and edges

**Node: state in, partial update out. Edge: what runs next.**

- reducers are not a messages feature
- two fields, two reducers: `text` replaced, `trail` collected


In [ ]:
class Note(TypedDict):
    text: str  # no reducer, so every write replaces
    trail: Annotated[list[str], operator.add]  # reducer, so every write concatenates


def shout(state: Note) -> dict:
    return {"text": state["text"].upper(), "trail": ["shout"]}


def frame(state: Note) -> dict:
    return {"text": f"<<{state['text']}>>", "trail": ["frame"]}


builder = StateGraph(Note)
builder.add_node("shout", shout)  # a plain function under a name
builder.add_node("frame", frame)
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)
notes = builder.compile()  # nothing runs before compile; compile checks the wiring

print(notes.get_graph().draw_mermaid())  # _png posts it to mermaid.ink; text is offline

print(
    notes.invoke(
        {"text": "a graph is a dict and some functions", "trail": []},  # every key up front
        config={"recursion_limit": 5},
    )
)


**Second silent failure: a node writes a key the schema lacks.**

- `TypedDict` is a hint for your type checker, not a runtime check
- `trail` comes back with one entry and nothing was raised
- a field mysteriously never set? grep it, count the spellings


In [ ]:
def frame_typo(state: Note) -> dict:
    return {"text": f"<<{state['text']}>>", "trial": ["frame"]}  # trial, not trail


builder = StateGraph(Note)
builder.add_node("shout", shout)
builder.add_node("frame", frame_typo)  # the only line that differs from before
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)

print(
    builder.compile().invoke(
        {"text": "one letter", "trail": []},
        config={"recursion_limit": 5},
    )
)


### The model in a node

**Same object session 1 ended on. Now it lives in `messages`.**

- `bind_tools` attaches JSON schemas; their source is session 3
- two dependent tools: the id must survive into the next turn
- a docstring naming the other tool is how order is taught


In [ ]:
from langchain_core.tools import tool

# a fake database: no network, same result every run
SPECIES = {"monstera": "mon-01", "ficus": "fic-02", "basil": "bas-03", "cactus": "cac-04"}
CARE = {
    "mon-01": "Water every 7 days. Bright indirect light. Wipe the leaves monthly.",
    "fic-02": "Water every 10 days. Hates being moved. Drops leaves when it sulks.",
    "bas-03": "Water daily. Full sun. Pinch the flowers off to keep the leaves coming.",
    "cac-04": "Water every 21 days in summer, never in winter. Full sun.",
}


@tool  # the docstring below is the description the model reads
def find_species(name: str) -> str:
    """Look up the species id of a house plant by its common name."""
    key = name.strip().lower()
    if key not in SPECIES:
        # an error the model can act on, not an exception
        return f"Unknown plant {name!r}. Known names: {', '.join(sorted(SPECIES))}."
    return SPECIES[key]


@tool
def care_notes(species_id: str) -> str:
    """Return watering and light instructions for a species id from find_species."""
    if species_id not in CARE:
        # where a guessed id lands; the model recovers next turn
        return f"No care notes for {species_id!r}. Call find_species first to get a valid id."
    return CARE[species_id]


TOOLS = [find_species, care_notes]
print([t.name for t in TOOLS])  # the tool name is the function name


In [ ]:
QUESTION = "My monstera looks sad. How often should I be watering it?"

model = chat_model("strong")
bound = model.bind_tools(TOOLS)  # a new object; nothing is sent here

ai = bound.invoke([{"role": "user", "content": QUESTION}])
print("type:      ", type(ai).__name__)
print("content:   ", repr(ai.content))  # usually empty on a tool-calling turn
print("tool_calls:", ai.tool_calls)  # the id pairs the result back to this call


**The model asked. Nothing has run.**

- one request, one turn; a plain dict works as a `HumanMessage`
- the id is load-bearing: the tool result must carry it back


### The conditional edge, by hand

**A function from state to the name of the next node.**

- six lines, no idea beyond that sentence
- the destination list constrains the drawing, not the routing
- no `recursion_limit`: a runaway loop bills a paid endpoint


In [ ]:
from langgraph.prebuilt import ToolNode


def call_model(state: MessagesState) -> dict:
    # Return the delta. A whole list here appends history to history.
    return {"messages": [bound.invoke(state["messages"])]}


def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    if last.tool_calls:  # the whole routing decision, one attribute
        return "tools"
    return END  # a legal destination too, and it stops the run


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))  # one node, every tool, picked by name
builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue, ["tools", END])  # honest diagram
builder.add_edge("tools", "model")  # this edge, and only this edge, is the loop
manual = builder.compile()

print(manual.get_graph().draw_mermaid())

result = manual.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},  # langgraph 1.2 defaults to 10007, not 25
)
print(len(result["messages"]), "messages")
print(result["messages"][-1].content)


**Those six lines ship prebuilt, as `tools_condition`.**

- buys: two lines instead of eight
- costs: no hits in the docs, whose canonical loop is hand-rolled
- its own docstring ships two imports that raise `ImportError`


In [ ]:
from langgraph.prebuilt import tools_condition

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
# tools_condition returns Literal["tools", "__end__"], so this name is fixed
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
graph = builder.compile()  # the graph the rest of the notebook runs

print(graph.get_graph().draw_mermaid())


**Rename the tool node and `tools_condition` breaks.**

- it still returns `"tools"`, and no node answers to that name
- a `ValueError` at compile time, so you find it now, not later


In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("plant_tools", ToolNode(TOOLS))  # the only change in the cell
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("plant_tools", "model")  # the loop edge is fine; the router is not

try:
    builder.compile()
except ValueError as error:
    print("ValueError:", error)


### ToolNode and the whole loop

**`ToolNode` is the session-1 block you wrote by hand.**

- reads `tool_calls`, runs each, returns a `ToolMessage` per call


In [ ]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)

for message in result["messages"]:  # role, tool call, and the id each answers
    message.pretty_print()


**Six messages. The loop ran twice and you wrote no loop.**

- 1 human; 2 AI `find_species("monstera")`; 3 tool `mon-01`
- 4 AI `care_notes("mon-01")`: that argument came from message 3
- 5 tool notes; 6 AI answer, no calls, routed to `END`
- the id pairing 2 with 3 is what you will read in every trace from now on


**Two things go wrong inside `ToolNode`, and not symmetrically.**

- a wrong tool name comes back as a `ToolMessage`, not an exception
- it lists the names that would have worked: the loop repairs itself


In [ ]:
def hallucinate(state: MessagesState) -> dict:
    # a fake model node: no request goes out
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "findSpecies",  # no such tool
                        "args": {"name": "monstera"},
                        "id": "call-1",  # normally the provider sets this
                    }
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", hallucinate)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_edge("model", "tools")  # a plain edge: straight into the tools
builder.add_edge("tools", END)

out = builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
print(out["messages"][-1].content)


In [ ]:
from langchain_core.tools import tool


@tool
def flaky(species_id: str) -> str:
    """Looks like any other tool right up until the day it does not."""
    raise TimeoutError("upstream took too long")  # stands in for an HTTP call


def call_flaky(state: MessagesState) -> dict:
    # the same fake model, aimed at the tool that throws
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {"name": "flaky", "args": {"species_id": "mon-01"}, "id": "call-1"}
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", call_flaky)
builder.add_node("tools", ToolNode([flaky]))  # only the throwing tool is registered
builder.add_edge(START, "model")
builder.add_edge("model", "tools")
builder.add_edge("tools", END)

try:
    builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
except TimeoutError as error:
    print("the run is over:", type(error).__name__, error)  # ToolNode re-raised it


**A wrong tool name is handled. An exception is not.**

- `ToolNode` re-raises anything but its own invocation error
- one timeout ends the run and takes the conversation with it
- your tool catches, and returns a string saying what to try instead


## init_chat_model

**Same `.env`, same `MODEL_CHEAP` and `MODEL_STRONG`. New return value.**

- a model object with `bind_tools`, not a client you post dicts to
- the size only picks a different id out of `.env`
- `ai` is the reply object from earlier in this notebook


In [ ]:
cheap = chat_model("cheap")
strong = chat_model("strong")
print(type(cheap).__name__, "|", type(strong).__name__)  # same class both times

print("id:               ", ai.id)  # set by the provider, not by us
print("additional_kwargs:", list(ai.additional_kwargs))  # provider-specific extras
print("response_metadata:", list(ai.response_metadata))  # finish reason, token counts


**Append the model's reply object itself. Never rebuild it.**

- providers hang reasoning signatures, cache markers and ids on it
- a rebuild prints identically, then fails on the next turn
- `id`, `additional_kwargs`, `response_metadata`: a rebuild drops all three


## Practice

**Port your assistant onto a graph, in your own repository.**

1. `MessagesState`, model node, `ToolNode`, one conditional edge
2. `should_continue` by hand, then swap in `tools_condition`
3. session-1 tool becomes `@tool` with a real docstring: that text is what the model reads
4. delete `parse_action` and the format prompt. Delete, not comment out


**Independent tools never make the loop come back. Nothing gets tested.**

5. a second and a third tool, one taking another's output: a lookup returning an id, and something that takes that id
6. every `invoke` passes a small `recursion_limit`
7. commit `runs/session-02.md`: the required-artifact card after the Langfuse section lists what goes in
8. ask early: today's three silent failures look like a stupid model

The notebook is a reference, not a source: your domain differs and the code will not copy across.

## Observability

**`print` stops paying for itself around the third node.**

- Langfuse in Docker on your own machine: no signup, no region, no quota
- a trace per run, a span per node, the real message list, how long each step took
- the traces never leave your laptop


Next to your `.env`:

```
curl -sSLO https://raw.githubusercontent.com/langfuse/langfuse/v3.224.0/docker-compose.yml
```

Then `docker-compose.override.yml` beside it. Compose applies it automatically, no `-f`.

```yaml
services:
  langfuse-worker:
    image: docker.io/langfuse/langfuse-worker:3.224.0
  langfuse-web:
    image: docker.io/langfuse/langfuse:3.224.0
    environment:
      LANGFUSE_INIT_ORG_ID: agents-course
      LANGFUSE_INIT_ORG_NAME: LLM Agents Course
      LANGFUSE_INIT_PROJECT_ID: assistant
      LANGFUSE_INIT_PROJECT_NAME: My assistant
      LANGFUSE_INIT_PROJECT_PUBLIC_KEY: ${LANGFUSE_PUBLIC_KEY}
      LANGFUSE_INIT_PROJECT_SECRET_KEY: ${LANGFUSE_SECRET_KEY}
      LANGFUSE_INIT_USER_EMAIL: ${LANGFUSE_USER_EMAIL}
      LANGFUSE_INIT_USER_NAME: Student
      LANGFUSE_INIT_USER_PASSWORD: ${LANGFUSE_USER_PASSWORD}
  postgres:                       # langfuse's own database, not the one you add later
    environment:
      POSTGRES_PASSWORD: ${LANGFUSE_POSTGRES_PASSWORD:-postgres}
```

- the upstream compose floats `:3`; the pins hold both services at the tested version
- the last two lines matter from session 8 on: without them the `POSTGRES_PASSWORD`
  you add for your own database is handed to langfuse's, whose volume already exists
- five new lines in `.env`: `LANGFUSE_HOST=http://localhost:3000`, your user email, plus three strings you invent for the two keys and your password
- keep the `pk-lf-` and `sk-lf-` prefixes
- never quote a `LANGFUSE_INIT_*` value: quoted, it is read literally, init silently does nothing, and you get an empty Langfuse whose keys are rejected

```
docker compose up -d
```

The first up pulls images and is slow; every later start is quick.

- compose and your code read the same `.env`: both sides agree by construction

**One handler in the config dict. Nothing inside the graph changes.**

- print the host, never a key: this file gets committed
- nothing on the second run? your keys and the container's drifted; `check_env.py` says so


In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # one handler is enough; every invoke opens its own trace
traced = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8, "callbacks": [handler]},
)

client.flush()  # a notebook kernel never exits, so nothing sends without this

print(traced["messages"][-1].content)

**Find three things in the trace and name each out loud.**

- the `tools` span, with both tool results inside it
- the second model call: history appended, not replaced
- the token counts and the per-step latency, session 3


**Required artifact: `runs/session-02.md`, committed.**

- the `draw_mermaid` diagram of your graph
- the six-plus-message dialogue whose fourth message cites the third
- the exported trace showing the pass through the nodes and a tool call

## Discussion

**You built the same agent twice and the second is longer.**

- where does the graph pay, and where is the `for` loop right?
- what separates them: tool count, pausing, another reader?
- argue for the loop: one call, no branching, a straight line


---

## Not taught today

**A conditional edge does not have to route to a tool.**

- it routes on anything computable from state, a classifier included
- `should_continue` with more return values, not a new pattern
- read-only: session 9 puts it beside four other shapes


In [ ]:
class Routed(TypedDict):
    text: str
    kind: str  # written by classify, read by route
    reply: str


def classify(state: Routed) -> dict:
    text = state["text"].lower()
    if text.endswith("?"):
        return {"kind": "question"}
    if text.startswith(("do ", "make ", "send ")):
        return {"kind": "command"}
    return {"kind": "other"}


def route(state: Routed) -> str:
    return state["kind"]  # same signature as should_continue


# builder.add_conditional_edges("classify", route, ["question", "command", "other"])


## Next time

**The graph becomes one call: create_agent, middleware, and tool design.**